# ABC Bank CLU - Evaluate Test Results

This notebook:
- reads the CLU test results CSV from the deploy/test notebook
- builds a cleaner evaluation summary
- produces simple charts for presentation and reporting
- highlights failed cases and likely improvement areas

In [ ]:
# %pip install pandas matplotlib python-dotenv

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv(".env")

PROJECT_NAME = os.getenv("AZURE_CLU_PROJECT_NAME", "abc-bank-clu")
DEPLOYMENT_NAME = os.getenv("AZURE_CLU_DEPLOYMENT_NAME", "production")

# default path from the deploy/test notebook
RESULTS_PATH = Path("analysis_output") / "abc_bank_clu_test_results.csv"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Test results CSV not found at: {RESULTS_PATH.resolve()}\n"
        "Run the deploy/test notebook first, or update RESULTS_PATH."
    )

print("Project:", PROJECT_NAME)
print("Deployment:", DEPLOYMENT_NAME)
print("Results file:", RESULTS_PATH.resolve())

In [ ]:
# =========================
# LOAD RESULTS
# =========================
df = pd.read_csv(RESULTS_PATH)

print("Rows loaded:", len(df))
df.head()

In [ ]:
# =========================
# NORMALIZE BOOLEAN COLUMNS
# =========================
bool_cols = ["intent_pass", "entity_category_pass", "entity_text_pass", "overall_pass"]

for col in bool_cols:
    if col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip().str.lower().map({
                "true": True,
                "false": False
            }).fillna(False)
        else:
            df[col] = df[col].astype(bool)

df[bool_cols].head()

In [ ]:
# =========================
# HIGH-LEVEL SUMMARY
# =========================
summary = {
    "project_name": PROJECT_NAME,
    "deployment_name": DEPLOYMENT_NAME,
    "total_tests": int(len(df)),
    "overall_pass_count": int(df["overall_pass"].sum()),
    "overall_fail_count": int((~df["overall_pass"]).sum()),
    "overall_pass_rate_pct": round(float(df["overall_pass"].mean() * 100), 2),
    "intent_pass_rate_pct": round(float(df["intent_pass"].mean() * 100), 2),
    "entity_category_pass_rate_pct": round(float(df["entity_category_pass"].mean() * 100), 2),
    "entity_text_pass_rate_pct": round(float(df["entity_text_pass"].mean() * 100), 2),
    "average_top_intent_score": round(float(df["top_intent_score"].fillna(0).mean()), 4),
}

df_summary = pd.DataFrame([summary])
df_summary

In [ ]:
# =========================
# PER-INTENT SUMMARY
# =========================
per_intent = (
    df.groupby("expected_intent", dropna=False)
      .agg(
          total_tests=("expected_intent", "size"),
          intent_pass_count=("intent_pass", "sum"),
          overall_pass_count=("overall_pass", "sum"),
          avg_top_intent_score=("top_intent_score", "mean"),
      )
      .reset_index()
)

per_intent["intent_pass_rate_pct"] = (per_intent["intent_pass_count"] / per_intent["total_tests"] * 100).round(2)
per_intent["overall_pass_rate_pct"] = (per_intent["overall_pass_count"] / per_intent["total_tests"] * 100).round(2)
per_intent["avg_top_intent_score"] = per_intent["avg_top_intent_score"].round(4)

per_intent

In [ ]:
# =========================
# FAILURE BREAKDOWN
# =========================
failure_breakdown = pd.DataFrame([
    {
        "check": "Intent",
        "fail_count": int((~df["intent_pass"]).sum())
    },
    {
        "check": "Entity Category",
        "fail_count": int((~df["entity_category_pass"]).sum())
    },
    {
        "check": "Entity Text",
        "fail_count": int((~df["entity_text_pass"]).sum())
    },
    {
        "check": "Overall",
        "fail_count": int((~df["overall_pass"]).sum())
    },
])

failure_breakdown

In [ ]:
# =========================
# FAILED CASES
# =========================
failed_cases = df.loc[~df["overall_pass"]].copy()

cols_to_show = [
    "utterance",
    "expected_intent",
    "actual_top_intent",
    "top_intent_score",
    "expected_entity_categories",
    "actual_entity_categories",
    "expected_entity_texts",
    "actual_entity_texts",
    "intent_pass",
    "entity_category_pass",
    "entity_text_pass",
    "overall_pass",
]

failed_cases[cols_to_show]

In [ ]:
# =========================
# ENHANCED FAILURE NOTES
# =========================
def derive_failure_reason(row):
    reasons = []
    if not row["intent_pass"]:
        reasons.append("Wrong intent")
    if not row["entity_category_pass"]:
        reasons.append("Missing/wrong entity category")
    if not row["entity_text_pass"]:
        reasons.append("Missing/wrong entity text span")
    return "; ".join(reasons) if reasons else "Passed"

df_detailed = df.copy()
df_detailed["failure_reason"] = df_detailed.apply(derive_failure_reason, axis=1)

df_detailed[[
    "utterance",
    "expected_intent",
    "actual_top_intent",
    "top_intent_score",
    "failure_reason",
    "overall_pass"
]]

In [ ]:
# =========================
# CHART 1 - OVERALL PASS/FAIL
# =========================
chart1 = df["overall_pass"].value_counts().rename(index={True: "Pass", False: "Fail"})

plt.figure(figsize=(7, 4))
chart1.plot(kind="bar")
plt.title("Overall CLU Test Results")
plt.xlabel("Result")
plt.ylabel("Number of Tests")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# CHART 2 - PASS RATES BY INTENT
# =========================
plt.figure(figsize=(8, 4))
plt.bar(per_intent["expected_intent"], per_intent["overall_pass_rate_pct"])
plt.title("Overall Pass Rate by Intent")
plt.xlabel("Intent")
plt.ylabel("Pass Rate (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# CHART 3 - AVERAGE CONFIDENCE BY INTENT
# =========================
plt.figure(figsize=(8, 4))
plt.bar(per_intent["expected_intent"], per_intent["avg_top_intent_score"])
plt.title("Average Top Intent Confidence by Intent")
plt.xlabel("Intent")
plt.ylabel("Average Confidence Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# CHART 4 - FAILURE TYPE COUNTS
# =========================
plt.figure(figsize=(8, 4))
plt.bar(failure_breakdown["check"], failure_breakdown["fail_count"])
plt.title("Failure Breakdown by Check Type")
plt.xlabel("Check")
plt.ylabel("Fail Count")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# EXECUTIVE SUMMARY TEXT
# =========================
summary_lines = []

summary_lines.append(f"Project: {PROJECT_NAME}")
summary_lines.append(f"Deployment: {DEPLOYMENT_NAME}")
summary_lines.append(f"Total test cases: {summary['total_tests']}")
summary_lines.append(f"Overall pass rate: {summary['overall_pass_rate_pct']}%")
summary_lines.append(f"Intent accuracy pass rate: {summary['intent_pass_rate_pct']}%")
summary_lines.append(f"Entity category pass rate: {summary['entity_category_pass_rate_pct']}%")
summary_lines.append(f"Entity text pass rate: {summary['entity_text_pass_rate_pct']}%")
summary_lines.append(f"Average top intent confidence: {summary['average_top_intent_score']}")

best_intent = per_intent.sort_values(["overall_pass_rate_pct", "avg_top_intent_score"], ascending=[False, False]).iloc[0]
worst_intent = per_intent.sort_values(["overall_pass_rate_pct", "avg_top_intent_score"], ascending=[True, True]).iloc[0]

summary_lines.append(
    f"Best-performing intent: {best_intent['expected_intent']} "
    f"({best_intent['overall_pass_rate_pct']}% pass rate)"
)
summary_lines.append(
    f"Lowest-performing intent: {worst_intent['expected_intent']} "
    f"({worst_intent['overall_pass_rate_pct']}% pass rate)"
)

if len(failed_cases) == 0:
    summary_lines.append("No failed test cases were detected in the current evaluation set.")
else:
    top_reason = (
        df_detailed.loc[~df_detailed["overall_pass"], "failure_reason"]
        .value_counts()
        .idxmax()
    )
    summary_lines.append(f"Most common failure pattern: {top_reason}")

executive_summary = "\n".join(summary_lines)
print(executive_summary)

In [ ]:
# =========================
# IMPROVEMENT SUGGESTIONS
# =========================
suggestions = []

if (~df["intent_pass"]).sum() > 0:
    suggestions.append("Add more utterance variation for intents with misclassification, especially around overlapping banking phrases.")

if (~df["entity_category_pass"]).sum() > 0:
    suggestions.append("Add more labeled examples for account_type and amount so the model sees more phrasing and account-name variants.")

if (~df["entity_text_pass"]).sum() > 0:
    suggestions.append("Review span labeling consistency for amount and account_type, especially short forms like 'savings' versus 'savings account'.")

if failed_cases["top_intent_score"].fillna(1).lt(0.7).any():
    suggestions.append("Low-confidence cases suggest the model needs more examples for ambiguous utterances and edge cases.")

if not suggestions:
    suggestions.append("Current test set looks strong. Add harder edge cases, paraphrases, and negative cases to validate robustness.")

for i, suggestion in enumerate(suggestions, start=1):
    print(f"{i}. {suggestion}")

In [ ]:
# =========================
# OPTIONAL: SAVE EVALUATION OUTPUTS
# =========================
output_dir = Path("analysis_output")
output_dir.mkdir(exist_ok=True)

summary_path = output_dir / "abc_bank_clu_evaluation_summary.csv"
per_intent_path = output_dir / "abc_bank_clu_per_intent_summary.csv"
failed_cases_path = output_dir / "abc_bank_clu_failed_cases.csv"
detailed_path = output_dir / "abc_bank_clu_detailed_results.csv"
summary_txt_path = output_dir / "abc_bank_clu_executive_summary.txt"

df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
per_intent.to_csv(per_intent_path, index=False, encoding="utf-8-sig")
failed_cases.to_csv(failed_cases_path, index=False, encoding="utf-8-sig")
df_detailed.to_csv(detailed_path, index=False, encoding="utf-8-sig")

with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write(executive_summary)

print("Saved:")
print(summary_path.resolve())
print(per_intent_path.resolve())
print(failed_cases_path.resolve())
print(detailed_path.resolve())
print(summary_txt_path.resolve())